# Distant Supervision for Biomedical NER Pseudo-Labeling

This notebook performs distant supervision to extend the biomedical NER label schema with additional entity types.

The previous preprocessing notebook produced two main datasets:

1. **PubMed sentence-level corpus**  
   This corpus contains tokenized biomedical sentences from PubMed titles and abstracts. It does not contain manual NER annotations, so all initial labels are `O`.

2. **BC5CDR preprocessed dataset**  
   This dataset contains gold token-level annotations for `Chemical` and `Disease`.

In this notebook, external biomedical dictionaries are used to add pseudo-labels for:

1. `Gene`
2. `Virus`

The final output is a multi-entity biomedical NER dataset containing `Chemical`, `Disease`, `Gene`, and `Virus` labels.

## Notebook Objectives

The main objectives of this notebook are:

1. Load the preprocessed PubMed and BC5CDR datasets.
2. Build a `Gene` dictionary from HGNC gene symbols and aliases.
3. Add `Gene` pseudo-labels using dictionary-based distant supervision.
4. Build a `Virus` dictionary from NCBI Taxonomy.
5. Add `Virus` pseudo-labels using dictionary-based distant supervision.
6. Preserve BC5CDR gold labels for `Chemical` and `Disease`.
7. Audit pseudo-label quality and gold-label overwrite safety.
8. Save the final pseudo-labeled datasets for downstream NER training.

The pseudo-labeled datasets generated here will later be used to train a biomedical NER model and support knowledge graph construction for drug repurposing analysis.


## Final Entity Label Schema

The final NER schema in this project contains four biomedical entity types:

1. `Chemical`
2. `Disease`
3. `Gene`
4. `Virus`

The complete BIO label set is:

```text
O
B-Chemical
I-Chemical
B-Disease
I-Disease
B-Gene
I-Gene
B-Virus
I-Virus
```

The label sources are different for each dataset.

For **BC5CDR**, the `Chemical` and `Disease` labels are gold annotations from the original dataset. These labels must be preserved and must not be overwritten during distant supervision.

For **PubMed**, the scraped corpus does not contain manual NER annotations. Therefore, the initial label for every token is `O`. The `Gene` and `Virus` labels are later added as pseudo-labels using dictionary-based distant supervision.

In this notebook, pseudo-labeling is applied conservatively. A new `Gene` or `Virus` label is only assigned to tokens that are still labeled as `O`. This rule protects existing gold labels in BC5CDR and prevents previously assigned pseudo-labels from being overwritten.

The final label column produced by this notebook is:

```text
decoded_tags_pseudo_final
```

This column will be used as the main label sequence for downstream biomedical NER training.



## Import Libraries

This section imports the required libraries for data loading, dictionary processing, text tokenization, pseudo-labeling, auditing, and saving the final datasets.

The notebook uses `pandas` for tabular data handling, `re` for biomedical tokenization, `tarfile` and `urllib` for downloading and extracting NCBI Taxonomy data, and `tqdm` to monitor long pseudo-labeling operations.

In [1]:
import os
import re
import tarfile
import urllib.request
from collections import Counter, defaultdict

import pandas as pd
from tqdm.auto import tqdm

d:\AI_Journey\NLP\nlp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

This section defines all input paths, output paths, and external resource locations used in the distant supervision pipeline.

The input datasets come from the previous EDA and preprocessing notebook. The HGNC file provides gene symbols and aliases, while the NCBI Taxonomy dump provides virus names and taxonomy information.

All output files generated by this notebook are saved into a dedicated output directory to keep the project structure organized.

In [ ]:
# Input paths from the preprocessing notebook
PUBMED_PREPROCESSED_PATH = "preprocessed_data/pubmed/pubmed_2022_2026_sentence_level_preprocessed.jsonl"

BC5CDR_PREPROCESSED_PATHS = {
    "train": "preprocessed_data/bc5cdr/bc5cdr_train_preprocessed.jsonl",
    "validation": "preprocessed_data/bc5cdr/bc5cdr_validation_preprocessed.jsonl",
    "test": "preprocessed_data/bc5cdr/bc5cdr_test_preprocessed.jsonl"
}

# External dictionary resource paths
# HGNC complete gene set URL
HGNC_URL = "https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/hgnc_complete_set.txt"

# NCBI Taxonomy resource configuration
NCBI_TAXDUMP_URL = "https://ftp.ncbi.nlm.nih.gov/pub/taxonomy/taxdump.tar.gz"
NCBI_TAXDUMP_PATH = "ncbi_taxdump.tar.gz"
NCBI_TAXDUMP_DIR = "ncbi_taxdump"

# Output directory
OUTPUT_DIR = "pseudolabeled_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Final output paths
PUBMED_FINAL_JSONL_PATH = os.path.join(
    OUTPUT_DIR,
    "pubmed_2022_2026_pseudolabel_gene_virus_final_no_covid19.jsonl"
)

PUBMED_FINAL_CSV_PATH = os.path.join(
    OUTPUT_DIR,
    "pubmed_2022_2026_pseudolabel_gene_virus_final_no_covid19.csv"
)

BC5CDR_FINAL_OUTPUT_PATHS = {
    split: os.path.join(
        OUTPUT_DIR,
        f"bc5cdr_{split}_pseudolabel_gene_virus_final_no_covid19.jsonl"
    )
    for split in ["train", "validation", "test"]
}

print("PubMed input path:", PUBMED_PREPROCESSED_PATH)
print("HGNC path:", HGNC_URL)
print("Output directory:", OUTPUT_DIR)

## Load Preprocessed PubMed Dataset

This section loads the sentence-level PubMed dataset generated by the previous preprocessing notebook.

At this stage, PubMed is an unlabeled corpus. Each row represents one sentence, with a token sequence and an initial all-`O` BIO tag sequence stored in the `decoded_tags` column.

This dataset will receive `Gene` and `Virus` pseudo-labels through distant supervision.

In [ ]:
pubmed_sentences_df = pd.read_json(
    PUBMED_PREPROCESSED_PATH,
    lines=True
)

print("PubMed sentence-level dataset shape:", pubmed_sentences_df.shape)
pubmed_sentences_df.head()

## Load Preprocessed BC5CDR Splits

This section loads the preprocessed BC5CDR train, validation, and test splits.

BC5CDR already contains gold BIO labels for `Chemical` and `Disease` in the `decoded_tags` column. These labels must be preserved during distant supervision. The pseudo-labeling functions will only assign new labels to tokens that are still labeled as `O`.

In [ ]:
bc5cdr_dfs_clean = {}

for split, path in BC5CDR_PREPROCESSED_PATHS.items():
    df = pd.read_json(path, lines=True)
    bc5cdr_dfs_clean[split] = df
    
    print(split, df.shape)

## Validate Token and Label Alignment

This section validates whether every token sequence has the same length as its corresponding BIO tag sequence.

Token-label alignment is a critical requirement for NER datasets because each token must have exactly one label. If the number of tokens and labels does not match, the row cannot be safely used for pseudo-labeling or model training.

In [ ]:
def validate_token_label_alignment(df, tag_column, dataset_name):
    # This function checks whether every token sequence has the same length as its label sequence
    alignment_matches = df.apply(
        lambda row: len(row["tokens"]) == len(row[tag_column]),
        axis=1
    )
    
    invalid_count = (~alignment_matches).sum()
    
    print(dataset_name, "invalid alignment count:", invalid_count)
    
    if invalid_count > 0:
        raise ValueError(f"Invalid token-label alignment found in {dataset_name}.")


validate_token_label_alignment(
    df=pubmed_sentences_df,
    tag_column="decoded_tags",
    dataset_name="PubMed"
)

for split, df in bc5cdr_dfs_clean.items():
    validate_token_label_alignment(
        df=df,
        tag_column="decoded_tags",
        dataset_name=f"BC5CDR {split}"
    )

## Define Biomedical Tokenizer

This section defines a tokenizer for biomedical text and dictionary terms.

The same tokenizer is used for both corpus tokens and dictionary terms to keep the matching process consistent. Biomedical terms often contain hyphens, slashes, numbers, and Greek letters. Examples include `SARS-CoV-2`, `HIV-1`, `IL-6`, `ACE2`, and `RIG-I`.

Using a biomedical-aware tokenizer helps preserve these terms as meaningful token units during dictionary matching.

In [ ]:
def biomedical_tokenize(sentence):
    # This function tokenizes biomedical text while preserving biomedical terms
    token_pattern = r"""
        [A-Za-z0-9]+(?:[-_/][A-Za-z0-9]+)*
        | [α-ωΑ-Ω]+
        | [^\w\s]
    """
    
    tokens = re.findall(token_pattern, str(sentence), flags=re.VERBOSE)
    
    return tokens

## Define Trie-Based Dictionary Matching Functions

This section defines helper functions for dictionary-based pseudo-labeling.

The dictionary terms are first converted into token sequences. Then, the token sequences are stored in a Trie index. During matching, the Trie allows the algorithm to efficiently scan sentence tokens and identify dictionary terms without checking every dictionary entry one by one.

The pseudo-labeling function is conservative: it only assigns a new entity label when all tokens in the matched span are currently labeled as `O`. This protects existing gold labels and prevents previously assigned pseudo-labels from being overwritten.

In [ ]:
def build_term_token_dict(terms, tokenizer_func, case_sensitive=True):
    # This function converts text terms into token-sequence dictionary entries
    term_dict = {}
    
    for term in terms:
        tokens = tokenizer_func(term)
        
        if not tokens:
            continue
        
        if case_sensitive:
            key = tuple(tokens)
        else:
            key = tuple(token.lower() for token in tokens)
        
        term_dict[key] = term
    
    return term_dict


class TrieNode:
    # This class represents one node in the Trie index
    def __init__(self):
        self.children = {}
        self.entity = None


def build_trie(term_dict):
    # This function builds a Trie index from tokenized dictionary terms
    root = TrieNode()
    
    for term_tokens, original_term in term_dict.items():
        node = root
        
        for token in term_tokens:
            if token not in node.children:
                node.children[token] = TrieNode()
            
            node = node.children[token]
        
        node.entity = original_term
    
    return root


def apply_trie_bio_labels(tokens, existing_tags, trie_root, entity_label, case_sensitive=True):
    # This function applies dictionary-based BIO pseudo-labeling using a Trie
    new_tags = existing_tags.copy()
    n = len(tokens)
    
    if case_sensitive:
        match_tokens = tokens
    else:
        match_tokens = [token.lower() for token in tokens]
    
    i = 0
    
    while i < n:
        node = trie_root
        longest_match_end = None
        j = i
        
        while j < n and match_tokens[j] in node.children:
            node = node.children[match_tokens[j]]
            
            if node.entity is not None:
                longest_match_end = j + 1
            
            j += 1
        
        if longest_match_end is not None:
            if all(tag == "O" for tag in new_tags[i:longest_match_end]):
                new_tags[i] = f"B-{entity_label}"
                
                for k in range(i + 1, longest_match_end):
                    new_tags[k] = f"I-{entity_label}"
                
                i = longest_match_end
                continue
        
        i += 1
    
    return new_tags

## Load HGNC Gene Dictionary

This section loads the HGNC complete gene set directly from the official HGNC public download URL.

HGNC is used as the external knowledge source for `Gene` pseudo-labeling. The dictionary is built from official gene symbols, alias symbols, and previous symbols. These fields provide multiple surface forms that may appear in biomedical literature.

Loading HGNC from an online source makes the notebook more reproducible because the gene dictionary does not need to be manually downloaded before running the pipeline.

In [ ]:
hgnc_df = pd.read_csv(
    HGNC_URL,
    sep="\t",
    dtype=str
)

print("HGNC shape:", hgnc_df.shape)
print("HGNC columns:")
print(hgnc_df.columns.tolist())

## Extract Gene Terms from HGNC

This section extracts gene-related surface forms from the HGNC dataset.

The selected HGNC fields are:

1. `symbol`  
   The official approved gene symbol.

2. `alias_symbol`  
   Alternative symbols that may appear in biomedical literature.

3. `prev_symbol`  
   Previous gene symbols that may still appear in older publications or existing biomedical text.

Some HGNC fields contain multiple values separated by the `|` character. These values are split and added individually to the gene term set.

The resulting term set will be used as the raw dictionary for `Gene` pseudo-labeling.

In [ ]:
def split_hgnc_multi_value(value):
    # This function splits HGNC multi-value fields separated by "|"
    if pd.isna(value):
        return []
    
    value = str(value).strip()
    
    if value == "":
        return []
    
    parts = value.split("|")
    parts = [part.strip() for part in parts if part.strip()]
    
    return parts


gene_terms = set()

for _, row in hgnc_df.iterrows():
    symbol = row.get("symbol")
    
    if pd.notna(symbol) and str(symbol).strip():
        gene_terms.add(str(symbol).strip())
    
    for alias in split_hgnc_multi_value(row.get("alias_symbol")):
        gene_terms.add(alias)
    
    for previous_symbol in split_hgnc_multi_value(row.get("prev_symbol")):
        gene_terms.add(previous_symbol)

print("Raw gene terms:", len(gene_terms))

## Extract Gene Terms from HGNC

This section extracts gene-related surface forms from the HGNC dataset.

The selected HGNC fields are:

1. `symbol`  
   The official approved gene symbol.

2. `alias_symbol`  
   Alternative symbols that may appear in biomedical literature.

3. `prev_symbol`  
   Previous gene symbols that may still appear in older publications or existing biomedical text.

Some HGNC fields contain multiple values separated by the `|` character. These values are split and added individually to the gene term set.

The resulting term set will be used as the raw dictionary for `Gene` pseudo-labeling.

In [ ]:
def split_hgnc_multi_value(value):
    # This function splits HGNC multi-value fields separated by "|"
    if pd.isna(value):
        return []
    
    value = str(value).strip()
    
    if value == "":
        return []
    
    parts = value.split("|")
    parts = [part.strip() for part in parts if part.strip()]
    
    return parts


gene_terms = set()

for _, row in hgnc_df.iterrows():
    symbol = row.get("symbol")
    
    if pd.notna(symbol) and str(symbol).strip():
        gene_terms.add(str(symbol).strip())
    
    for alias in split_hgnc_multi_value(row.get("alias_symbol")):
        gene_terms.add(alias)
    
    for previous_symbol in split_hgnc_multi_value(row.get("prev_symbol")):
        gene_terms.add(previous_symbol)

print("Raw gene terms:", len(gene_terms))

## Filter Gene Terms

This section filters raw HGNC gene terms before pseudo-labeling.

Although HGNC is a reliable biomedical resource, dictionary-based matching can still create false positives. Some gene symbols and aliases overlap with common English words, clinical abbreviations, or biomedical terms that do not always refer to genes.

The filtering process removes:

1. Very short terms.
2. Pure numeric terms.
3. Common stopwords.
4. Extremely long multi-word terms.
5. Manually identified ambiguous biomedical abbreviations.

This step helps reduce obvious false positives while preserving useful gene surface forms for distant supervision.

In [ ]:
GENE_STOPWORDS = {
    "A", "AN", "AND", "AS", "AT", "BE", "BY", "FOR", "FROM", "IN", "INTO",
    "IS", "IT", "OF", "ON", "OR", "THE", "TO", "VIA", "VS", "WITH",
    "SET", "WAS", "CAN", "MAY", "NOT", "NO", "YES", "IF", "THEN",
    "CAT", "DOG", "MARCH", "SEP", "SEPT"
}

GENE_BLACKLIST = {
    "ART",
    "SARS",
    "ROS",
    "MRI",
    "PCI",
    "CPAP",
    "BMD",
    "DBS",
    "TDI",
    "GFR",
    "ALT",
    "OCT",
    "NET",
    "CTA",
    "CIN"
}

GENE_STOPWORDS_UPPER = {term.upper() for term in GENE_STOPWORDS}
GENE_BLACKLIST_UPPER = {term.upper() for term in GENE_BLACKLIST}


def is_valid_gene_term(term):
    # This function filters noisy or ambiguous HGNC gene terms
    term = str(term).strip()
    term_upper = term.upper()
    
    if len(term) < 3:
        return False
    
    if term_upper in GENE_STOPWORDS_UPPER:
        return False
    
    if term_upper in GENE_BLACKLIST_UPPER:
        return False
    
    if term.isdigit():
        return False
    
    if len(term.split()) > 5:
        return False
    
    return True


gene_terms_clean = sorted(
    set(term for term in gene_terms if is_valid_gene_term(term)),
    key=len,
    reverse=True
)

print("Clean gene terms:", len(gene_terms_clean))
gene_terms_clean[:30]

## Gene Filtering Decision

The gene dictionary is filtered because exact dictionary matching does not understand context.

For example, some HGNC aliases can also refer to clinical procedures, imaging methods, measurements, diseases, or general biomedical concepts. If these terms are kept without filtering, the distant supervision process may label many non-gene mentions as `Gene`.

Examples of ambiguous terms removed through the manual blacklist include:

1. `ART`, which may refer to antiretroviral therapy.
2. `SARS`, which may refer to severe acute respiratory syndrome.
3. `ROS`, which may refer to reactive oxygen species.
4. `MRI`, which refers to magnetic resonance imaging.
5. `GFR`, which refers to glomerular filtration rate.
6. `ALT`, which may refer to alanine transaminase in clinical contexts.

This filtering step does not eliminate all possible noise, but it reduces obvious false positives and makes the pseudo-labeling process more defensible.

## Build Gene Dictionary and Gene Trie Index

This section converts the cleaned HGNC gene terms into a token-sequence dictionary and then builds a Trie index.

The gene dictionary is built using the biomedical tokenizer so that dictionary terms and corpus tokens follow the same tokenization logic.|

Gene matching is performed in a case-sensitive way because gene symbols are often case-sensitive and may overlap with common words or abbreviations when lowercased.

The Trie index allows efficient exact matching and supports longest-match behavior for multi-token gene terms.

In [ ]:
gene_term_dict = build_term_token_dict(
    gene_terms_clean,
    biomedical_tokenize,
    case_sensitive=True
)

gene_trie = build_trie(gene_term_dict)

print("Gene dictionary size:", len(gene_term_dict))
print("Gene trie built.")


## Apply Gene Pseudo-Labeling to PubMed

This section applies `Gene` pseudo-labeling to the PubMed sentence-level dataset.

The PubMed scraped corpus does not contain manual NER annotations. Therefore, every token initially has the label `O`. In this step, the cleaned HGNC gene dictionary and Gene Trie are used to detect gene mentions in each sentence.

When a token sequence matches a gene term from the Trie, the matched tokens are assigned BIO labels:

```text
B-Gene
I-Gene
```

The pseudo-labeling function only assigns a new label if the matched token span is still fully labeled as `O`. This conservative rule prevents existing labels from being overwritten and keeps the pseudo-labeling process consistent with the later BC5CDR processing.

The result is stored in a new column:

```text
decoded_tags_pseudo_gene
```

This column is an intermediate checkpoint. It contains the original PubMed labels plus the added `Gene` pseudo-labels, and it will be used as the input for the next pseudo-labeling stage: `Virus`.





## Apply Gene Pseudo-Labeling to BC5CDR

This section applies `Gene` pseudo-labeling to the preprocessed BC5CDR dataset.

Unlike PubMed, BC5CDR already contains gold annotations for `Chemical` and `Disease`. These gold labels are stored in the `decoded_tags` column and must not be overwritten during distant supervision.

The Gene Trie is used to detect gene mentions in each BC5CDR split: `train`, `validation`, and `test`. When a token sequence matches a gene term from the cleaned HGNC dictionary, the matched span is assigned `B-Gene` and `I-Gene` labels.

The pseudo-labeling function only assigns `Gene` labels to spans where all tokens are still labeled as `O`. This conservative rule preserves the original BC5CDR gold labels while extending the dataset with additional `Gene` pseudo-labels.

The result for each split is stored in:

```text
bc5cdr_dfs_pseudo_gene
```

Each DataFrame inside this dictionary contains a new column:

```text
decoded_tags_pseudo_gene
```

This column will be used as the input checkpoint for the next pseudo-labeling stage: `Virus`.





In [ ]:
bc5cdr_dfs_pseudo_gene = {}

for split, df in bc5cdr_dfs_clean.items():
    print(f"Processing split: {split}")
    
    df_gene = df.copy()
    
    df_gene["decoded_tags_pseudo_gene"] = df_gene.progress_apply(
        lambda row: apply_trie_bio_labels(
            tokens=row["tokens"],
            existing_tags=row["decoded_tags"],
            trie_root=gene_trie,
            entity_label="Gene",
            case_sensitive=True
        ),
        axis=1
    )
    
    bc5cdr_dfs_pseudo_gene[split] = df_gene
    
    print(split, df_gene.shape)


## Validate Gene Pseudo-Label Alignment

This section validates token-label alignment after `Gene` pseudo-labeling.

For every row, the number of tokens must be equal to the number of labels in `decoded_tags_pseudo_gene`. This validation is important because NER training requires one label for each token.

If any row has mismatched token and label lengths, the notebook will raise an error before moving to the next stage. This prevents invalid pseudo-labeled data from being used in the `Virus` pseudo-labeling step or downstream model training.

In [ ]:
validate_token_label_alignment(
    df=pubmed_sentences_df,
    tag_column="decoded_tags_pseudo_gene",
    dataset_name="PubMed after Gene pseudo-labeling"
)

for split, df in bc5cdr_dfs_pseudo_gene.items():
    validate_token_label_alignment(
        df=df,
        tag_column="decoded_tags_pseudo_gene",
        dataset_name=f"BC5CDR {split} after Gene pseudo-labeling"
    )

## Define Entity Extraction Helper Functions

This section defines helper functions to audit pseudo-labeling results at the entity level.

The pseudo-labeling process produces token-level BIO tags. However, for analysis and quality inspection, it is more useful to convert token-level labels into complete entity mentions.

For example, a sequence such as:

```text
["B-Gene", "I-Gene"]
```

should be counted as one complete `Gene` entity mention, not as two separate labeled tokens.

The helper functions in this section extract complete entity spans from BIO labels and convert them into a DataFrame that can be used for coverage analysis, top-entity inspection, and false-positive checking.


Selanjutnya kita buat **Gene entity DataFrame** untuk PubMed dan BC5CDR, lalu audit coverage dan top Gene mentions.


In [ ]:
def extract_entities_from_bio(tokens, tags):
    # This function extracts entity spans from tokens and BIO tags
    entities = []
    current_entity = None
    
    for idx, (token, tag) in enumerate(zip(tokens, tags)):
        if tag == "O":
            if current_entity is not None:
                entities.append(current_entity)
                current_entity = None
            continue
        
        if "-" not in tag:
            continue
        
        prefix, entity_type = tag.split("-", 1)
        
        if prefix == "B":
            if current_entity is not None:
                entities.append(current_entity)
            
            current_entity = {
                "entity_type": entity_type,
                "start_token": idx,
                "end_token": idx + 1,
                "tokens": [token]
            }
        
        elif prefix == "I":
            if current_entity is not None and current_entity["entity_type"] == entity_type:
                current_entity["end_token"] = idx + 1
                current_entity["tokens"].append(token)
            else:
                current_entity = {
                    "entity_type": entity_type,
                    "start_token": idx,
                    "end_token": idx + 1,
                    "tokens": [token]
                }
    
    if current_entity is not None:
        entities.append(current_entity)
    
    for entity in entities:
        entity["entity_text"] = " ".join(entity["tokens"])
    
    return entities


def build_entity_dataframe(df, tag_column, dataset_name, split=None):
    # This function converts token-level BIO annotations into an entity-level DataFrame
    entity_rows = []
    
    for row_idx, row in df.iterrows():
        entities = extract_entities_from_bio(
            tokens=row["tokens"],
            tags=row[tag_column]
        )
        
        for entity in entities:
            entity_rows.append({
                "dataset": dataset_name,
                "split": split,
                "row_idx": row_idx,
                "entity_type": entity["entity_type"],
                "entity_text": entity["entity_text"],
                "start_token": entity["start_token"],
                "end_token": entity["end_token"],
                "sentence_text": row.get("sentence_text", ""),
                "pmid": row.get("pmid", None),
                "publication_year": row.get("publication_year", None),
                "url": row.get("url", None)
            })
    
    return pd.DataFrame(entity_rows)

## Build Gene Entity DataFrames

This section builds entity-level DataFrames after `Gene` pseudo-labeling.

For PubMed, the entity DataFrame is built from the `decoded_tags_pseudo_gene` column. Since PubMed started with all-`O` labels, the extracted `Gene` entities represent pseudo-labeled mentions from the HGNC dictionary.

For BC5CDR, the entity DataFrames are also built from `decoded_tags_pseudo_gene`. These DataFrames contain the original gold `Chemical` and `Disease` entities plus the newly added `Gene` pseudo-labels.

The resulting entity-level DataFrames are used for coverage analysis and manual inspection of pseudo-label quality.

In [ ]:
pubmed_entities_gene_df = build_entity_dataframe(
    df=pubmed_sentences_df,
    tag_column="decoded_tags_pseudo_gene",
    dataset_name="pubmed"
)

bc5cdr_entities_gene_dfs = {}

for split, df in bc5cdr_dfs_pseudo_gene.items():
    bc5cdr_entities_gene_dfs[split] = build_entity_dataframe(
        df=df,
        tag_column="decoded_tags_pseudo_gene",
        dataset_name="bc5cdr",
        split=split
    )

bc5cdr_entities_gene_df = pd.concat(
    bc5cdr_entities_gene_dfs.values(),
    ignore_index=True
)

print("PubMed entity DataFrame shape:", pubmed_entities_gene_df.shape)
print("BC5CDR entity DataFrame shape:", bc5cdr_entities_gene_df.shape)

## Audit Gene Pseudo-Label Coverage

This section audits how many `Gene` pseudo-labels were added to PubMed and BC5CDR.

The audit focuses on:

1. The number of sentences containing at least one `Gene` label.
2. The percentage of sentences containing at least one `Gene` label.
3. The total number of `Gene` entity mentions.
4. The number of unique `Gene` entity surface forms.

This coverage analysis helps evaluate whether the HGNC-based distant supervision process produced a reasonable amount of Gene annotations without overwhelming the dataset.

In [ ]:
def summarize_entity_coverage(df, entity_df, tag_column, entity_type, dataset_name):
    # This function summarizes sentence-level and entity-level coverage for one entity type
    sentence_has_entity = df[tag_column].apply(
        lambda tags: any(tag.endswith(f"-{entity_type}") for tag in tags)
    )
    
    entity_mentions_df = entity_df[
        entity_df["entity_type"] == entity_type
    ].copy()
    
    summary = {
        "dataset": dataset_name,
        "total_sentences": len(df),
        "sentences_with_entity": sentence_has_entity.sum(),
        "sentence_coverage_percentage": sentence_has_entity.mean() * 100,
        "entity_mentions": len(entity_mentions_df),
        "unique_entity_texts": entity_mentions_df["entity_text"].nunique()
    }
    
    return summary


gene_coverage_rows = []

gene_coverage_rows.append(
    summarize_entity_coverage(
        df=pubmed_sentences_df,
        entity_df=pubmed_entities_gene_df,
        tag_column="decoded_tags_pseudo_gene",
        entity_type="Gene",
        dataset_name="pubmed"
    )
)

for split, df in bc5cdr_dfs_pseudo_gene.items():
    split_entity_df = bc5cdr_entities_gene_df[
        bc5cdr_entities_gene_df["split"] == split
    ].copy()
    
    gene_coverage_rows.append(
        summarize_entity_coverage(
            df=df,
            entity_df=split_entity_df,
            tag_column="decoded_tags_pseudo_gene",
            entity_type="Gene",
            dataset_name=f"bc5cdr_{split}"
        )
    )

gene_coverage_df = pd.DataFrame(gene_coverage_rows)

gene_coverage_df

## Inspect Top Gene Mentions

This section inspects the most frequent `Gene` mentions produced by distant supervision.

Top-entity inspection is useful for identifying whether the pseudo-labeling results are biologically reasonable or dominated by noisy ambiguous terms. If many frequent terms appear to be false positives, the gene blacklist can be updated and the pseudo-labeling process can be rerun.

This step is part of pseudo-label quality control.

In [ ]:
top_pubmed_gene_mentions = (
    pubmed_entities_gene_df[pubmed_entities_gene_df["entity_type"] == "Gene"]
    .groupby("entity_text")
    .size()
    .reset_index(name="mention_count")
    .sort_values(by="mention_count", ascending=False)
)

top_pubmed_gene_mentions.head(30)

## Inspect Top Gene Mentions in BC5CDR

This section inspects the most frequent `Gene` pseudo-labels added to each BC5CDR split.

Because BC5CDR is originally focused on `Chemical` and `Disease`, the number of `Gene` mentions is expected to be smaller than in the PubMed corpus. This inspection helps verify that the added `Gene` pseudo-labels are not excessive and that the results remain plausible.

In [ ]:
top_bc5cdr_gene_mentions = (
    bc5cdr_entities_gene_df[bc5cdr_entities_gene_df["entity_type"] == "Gene"]
    .groupby(["split", "entity_text"])
    .size()
    .reset_index(name="mention_count")
    .sort_values(by=["split", "mention_count"], ascending=[True, False])
)

top_bc5cdr_gene_mentions.groupby("split").head(20)

## Audit Gold Label Overwrite After Gene Pseudo-Labeling

This section checks whether `Gene` pseudo-labeling overwrote any existing BC5CDR gold labels.

BC5CDR contains gold annotations for `Chemical` and `Disease`. These labels must remain unchanged after distant supervision. The expected result of this audit is zero overwritten gold-label tokens.

This audit is important because the pseudo-labeling function should only assign new labels to tokens that are still labeled as `O`.

In [ ]:
def count_changed_gold_labels(original_tags, pseudo_tags):
    # This function counts gold-label tokens that changed after pseudo-labeling
    changed = 0
    
    for original_tag, pseudo_tag in zip(original_tags, pseudo_tags):
        if original_tag != "O" and original_tag != pseudo_tag:
            changed += 1
    
    return changed


gene_gold_overwrite_rows = []

for split, df in bc5cdr_dfs_pseudo_gene.items():
    df_audit = df.copy()
    
    df_audit["gold_token_overwrites"] = df_audit.apply(
        lambda row: count_changed_gold_labels(
            original_tags=row["decoded_tags"],
            pseudo_tags=row["decoded_tags_pseudo_gene"]
        ),
        axis=1
    )
    
    gene_gold_overwrite_rows.append({
        "split": split,
        "sentences_with_gold_overwrite": (df_audit["gold_token_overwrites"] > 0).sum(),
        "total_gold_token_overwrites": df_audit["gold_token_overwrites"].sum()
    })

gene_gold_overwrite_df = pd.DataFrame(gene_gold_overwrite_rows)

gene_gold_overwrite_df

## Gene Pseudo-Labeling Audit Summary

At this stage, `Gene` pseudo-labeling has been applied and audited.

The audit checks whether the generated `Gene` labels are reasonable by inspecting coverage statistics, frequent entity mentions, and gold-label overwrite safety.

The BC5CDR overwrite audit is especially important because the original `Chemical` and `Disease` annotations are gold labels. The expected result is that no gold-label tokens are changed during `Gene` pseudo-labeling.

The next stage adds `Virus` pseudo-labels on top of the `decoded_tags_pseudo_gene` checkpoint.

## Download and Extract NCBI Taxonomy

This section downloads and extracts the NCBI Taxonomy dump.

NCBI Taxonomy is used as the external knowledge source for `Virus` pseudo-labeling. The taxonomy dump contains structured information about taxonomic nodes and scientific names, including virus names, synonyms, common names, and acronyms.

The downloaded archive contains two important files for this notebook:

1. `nodes.dmp`  
   Contains taxonomy node relationships, including parent-child relationships.

2. `names.dmp`  
   Contains scientific names, synonyms, common names, and other name variants for each taxon.

These files will be used to identify all taxonomic names under the `Viruses` root taxon.

In [ ]:
if not os.path.exists(NCBI_TAXDUMP_PATH):
    urllib.request.urlretrieve(
        NCBI_TAXDUMP_URL,
        NCBI_TAXDUMP_PATH
    )

os.makedirs(NCBI_TAXDUMP_DIR, exist_ok=True)

with tarfile.open(NCBI_TAXDUMP_PATH, "r:gz") as tar:
    tar.extractall(NCBI_TAXDUMP_DIR)

print("NCBI Taxonomy dump extracted to:", NCBI_TAXDUMP_DIR)

## Load NCBI Taxonomy Nodes

This section loads the `nodes.dmp` file from the NCBI Taxonomy dump.

The `nodes.dmp` file describes the taxonomy tree structure. Each row contains a taxon ID, its parent taxon ID, and its taxonomic rank.

This information is needed to collect all descendant taxa under the `Viruses` root taxon.

In [ ]:
nodes_path = os.path.join(NCBI_TAXDUMP_DIR, "nodes.dmp")

nodes_df = pd.read_csv(
    nodes_path,
    sep=r"\t\|\t",
    engine="python",
    header=None,
    usecols=[0, 1, 2],
    names=["tax_id", "parent_tax_id", "rank"],
    dtype=str
)

nodes_df["rank"] = nodes_df["rank"].str.replace(r"\t\|$", "", regex=True)

print("Nodes shape:", nodes_df.shape)
nodes_df.head()

## Extract Virus Taxonomy IDs

This section extracts all taxonomy IDs under the `Viruses` root taxon.

In NCBI Taxonomy, the root taxon ID for viruses is:

```text
10239
```

Using the parent-child relationships from `nodes.dmp`, this step traverses the taxonomy tree and collects all descendant taxon IDs under `Viruses`.

The resulting set of taxon IDs is used to filter virus-related names from `names.dmp`.




Selanjutnya setelah ini: **load names.dmp**, filter virus names, clean virus terms, dan tambah manual aliases.


In [ ]:
VIRUSES_TAX_ID = "10239"

parent_to_children = nodes_df.groupby("parent_tax_id")["tax_id"].apply(list).to_dict()


def get_descendant_taxids(root_tax_id, parent_to_children):
    # This function collects all descendant taxonomy IDs under a root taxon
    descendants = set()
    stack = [root_tax_id]
    
    while stack:
        current = stack.pop()
        
        if current in descendants:
            continue
        
        descendants.add(current)
        
        children = parent_to_children.get(current, [])
        stack.extend(children)
    
    return descendants


virus_taxids = get_descendant_taxids(
    root_tax_id=VIRUSES_TAX_ID,
    parent_to_children=parent_to_children
)

print("Virus taxid count:", len(virus_taxids))

## Load NCBI Taxonomy Names

This section loads the `names.dmp` file from the NCBI Taxonomy dump.

The `names.dmp` file contains different names associated with each taxon ID, such as scientific names, synonyms, common names, and acronyms.

After loading the file, the names will be filtered to keep only names whose taxon IDs belong to the `Viruses` taxonomy branch.

In [ ]:
names_path = os.path.join(NCBI_TAXDUMP_DIR, "names.dmp")

names_df = pd.read_csv(
    names_path,
    sep=r"\t\|\t",
    engine="python",
    header=None,
    usecols=[0, 1, 3],
    names=["tax_id", "name_txt", "name_class"],
    dtype=str
)

names_df["name_class"] = names_df["name_class"].str.replace(r"\t\|$", "", regex=True)

print("Names shape:", names_df.shape)
names_df.head()

## Filter Virus Names from NCBI Taxonomy

This section filters NCBI Taxonomy names to keep only virus-related names.

The filtering uses two conditions:

1. The `tax_id` must be part of the descendant taxon IDs under the `Viruses` root taxon.
2. The `name_class` must represent a useful name type for dictionary matching.

The selected name classes include scientific names, synonyms, equivalent names, common names, GenBank common names, and acronyms.

These names provide multiple surface forms that may appear in biomedical literature.

In [ ]:
allowed_name_classes = {
    "scientific name",
    "synonym",
    "equivalent name",
    "genbank common name",
    "common name",
    "acronym"
}

virus_names_df = names_df[
    names_df["tax_id"].isin(virus_taxids)
    & names_df["name_class"].isin(allowed_name_classes)
].copy()

print("Virus names shape:", virus_names_df.shape)
virus_names_df.head()

## Filter Clean Virus Terms

This section cleans the raw virus names extracted from NCBI Taxonomy.

NCBI Taxonomy is very detailed and contains many highly specific virus strain, isolate, and reassortant names. These terms are valid for taxonomy, but they are often too specific for abstract-level NER and can make the dictionary unnecessarily large.

The cleaning strategy removes:

1. Very short terms.
2. Generic terms such as `virus`, `viruses`, and `viral`.
3. Pure numeric terms.
4. Extremely long terms.
5. Highly strain-specific terms containing complex isolate or reassortant descriptions.
6. Terms that should not be labeled as `Virus`, such as `COVID-19`.

The term `COVID-19` is explicitly removed because it is a disease, not a virus. The virus responsible for COVID-19 is `SARS-CoV-2`.

In [ ]:
VIRUS_BLACKLIST = {
    "virus",
    "viruses",
    "viral",
    "phage",
    "phages",
    "viroid",
    "viroids",
    "unclassified",
    "unidentified",
    "environmental samples",
    "COVID-19",
    "other sequences"
}

VIRUS_BLACKLIST_LOWER = {term.lower() for term in VIRUS_BLACKLIST}


def is_valid_virus_term(term):
    # This function filters noisy or overly specific virus names
    term = str(term).strip()
    term_lower = term.lower()
    
    if len(term) < 4:
        return False
    
    if term_lower in VIRUS_BLACKLIST_LOWER:
        return False
    
    if term.isdigit():
        return False
    
    if len(term) > 80:
        return False
    
    if len(term.split()) > 8:
        return False
    
    if "reassortant" in term_lower:
        return False
    
    if term.count("/") >= 2:
        return False
    
    if "(" in term or ")" in term:
        return False
    
    return True


virus_terms_clean = sorted(
    set(
        term for term in virus_names_df["name_txt"].tolist()
        if is_valid_virus_term(term)
    ),
    key=len,
    reverse=True
)

print("Clean virus terms:", len(virus_terms_clean))
virus_terms_clean[:30]

## Add Manual Virus Aliases

This section adds manually selected virus aliases to the cleaned NCBI virus dictionary.

Although NCBI Taxonomy provides many virus names, biomedical abstracts often use short acronyms or common surface forms that may not always be captured in the desired form after filtering. Examples include `HIV`, `HBV`, `HCV`, `RSV`, `DENV`, and `ZIKV`.

Manual aliases are added to improve coverage for common virus mentions in biomedical literature.

The alias list intentionally excludes `COVID-19` because COVID-19 is a disease, not a virus. The corresponding virus entity is `SARS-CoV-2`.

In [ ]:
manual_virus_aliases = [
    "SARS-CoV-2",
    "HIV",
    "HIV-1",
    "HIV-2",
    "HBV",
    "HCV",
    "HPV",
    "HSV",
    "HSV-1",
    "HSV-2",
    "RSV",
    "IAV",
    "IBV",
    "DENV",
    "ZIKV",
    "EBOV",
    "ASFV",
    "PEDV",
    "PRRSV",
    "MERS-CoV",
    "SARS-CoV",
    "H1N1",
    "H3N2",
    "H5N1",
    "H7N9",
    "influenza virus",
    "influenza A virus",
    "influenza B virus",
    "dengue virus",
    "Zika virus",
    "Ebola virus",
    "hepatitis B virus",
    "hepatitis C virus",
    "herpes simplex virus",
    "respiratory syncytial virus",
    "human immunodeficiency virus",
    "severe acute respiratory syndrome coronavirus 2",
    "adenovirus",
    "rotavirus",
    "norovirus",
    "cytomegalovirus",
    "BK virus"
]

manual_virus_aliases = [
    term for term in manual_virus_aliases
    if term.lower() not in VIRUS_BLACKLIST_LOWER
]

virus_terms_clean = sorted(
    set(virus_terms_clean).union(manual_virus_aliases),
    key=len,
    reverse=True
)

print("Final virus terms:", len(virus_terms_clean))
virus_terms_clean[:30]

## Build Virus Dictionary and Virus Trie Index

This section converts the cleaned virus term list into a token-sequence dictionary and builds a Virus Trie index.

Virus matching is performed in a case-insensitive way because virus names often appear with different capitalization styles in biomedical text. For example, `Dengue virus` and `dengue virus` should be treated as the same surface form for matching.

The Virus Trie allows efficient exact matching of both single-token and multi-token virus names. It will be used to add `Virus` pseudo-labels on top of the previously generated `Gene` pseudo-label checkpoint.

In [ ]:
virus_term_dict = build_term_token_dict(
    virus_terms_clean,
    biomedical_tokenize,
    case_sensitive=False
)

virus_trie = build_trie(virus_term_dict)

print("Virus dictionary size:", len(virus_term_dict))
print("Virus trie built.")


## Apply Virus Pseudo-Labeling to PubMed

This section applies `Virus` pseudo-labeling to the PubMed sentence-level dataset.

The input label column for this step is:

```text
decoded_tags_pseudo_gene
```

This means that `Virus` pseudo-labeling is applied after the `Gene` pseudo-labeling checkpoint. If a virus term from the Virus Trie matches a token sequence, the matched span is assigned `B-Virus` and `I-Virus` labels.

The pseudo-labeling function only assigns a new label if all tokens in the matched span are still labeled as `O`. This prevents `Virus` labels from overwriting existing `Gene` labels.

The final result is stored in:

```text
decoded_tags_pseudo_final
```

This column contains the final PubMed pseudo-label sequence for downstream NER training.











In [ ]:
pubmed_sentences_df["decoded_tags_pseudo_final"] = pubmed_sentences_df.progress_apply(
    lambda row: apply_trie_bio_labels(
        tokens=row["tokens"],
        existing_tags=row["decoded_tags_pseudo_gene"],
        trie_root=virus_trie,
        entity_label="Virus",
        case_sensitive=False
    ),
    axis=1
)

print("Virus pseudo-labeling completed for PubMed.")

## Apply Virus Pseudo-Labeling to BC5CDR

This section applies `Virus` pseudo-labeling to each BC5CDR split.

The input label column is:

```text
decoded_tags_pseudo_gene
```

This ensures that `Virus` pseudo-labeling is applied on top of the `Gene` pseudo-label checkpoint.

Since BC5CDR already contains gold `Chemical` and `Disease` annotations, the pseudo-labeling function only assigns `Virus` labels to spans where all tokens are still labeled as `O`. This rule prevents `Virus` labels from overwriting gold labels or previously assigned `Gene` pseudo-labels.

The final label sequence for each split is stored in:

```text
decoded_tags_pseudo_final
```


In [ ]:
bc5cdr_dfs_pseudo_final = {}

for split, df in bc5cdr_dfs_pseudo_gene.items():
    print(f"Processing split: {split}")
    
    df_final = df.copy()
    
    df_final["decoded_tags_pseudo_final"] = df_final.progress_apply(
        lambda row: apply_trie_bio_labels(
            tokens=row["tokens"],
            existing_tags=row["decoded_tags_pseudo_gene"],
            trie_root=virus_trie,
            entity_label="Virus",
            case_sensitive=False
        ),
        axis=1
    )
    
    bc5cdr_dfs_pseudo_final[split] = df_final
    
    print(split, df_final.shape)


## Validate Final Pseudo-Label Alignment

This section validates token-label alignment after the final `Gene` and `Virus` pseudo-labeling steps.

The final label column is:

```text
decoded_tags_pseudo_final
```

For every row, the number of labels in this column must match the number of tokens. This validation ensures that the final pseudo-labeled datasets are safe for downstream NER training.

In [ ]:
validate_token_label_alignment(
    df=pubmed_sentences_df,
    tag_column="decoded_tags_pseudo_final",
    dataset_name="PubMed after final pseudo-labeling"
)

for split, df in bc5cdr_dfs_pseudo_final.items():
    validate_token_label_alignment(
        df=df,
        tag_column="decoded_tags_pseudo_final",
        dataset_name=f"BC5CDR {split} after final pseudo-labeling"
    )



## Build Final Entity DataFrames

This section builds entity-level DataFrames using the final pseudo-label column.

The final label column is:

```text
decoded_tags_pseudo_final
```

For PubMed, this column contains the original all-`O` labels plus the added `Gene` and `Virus` pseudo-labels.

For BC5CDR, this column contains the original gold `Chemical` and `Disease` labels plus the added `Gene` and `Virus` pseudo-labels.

The resulting entity-level DataFrames are used to audit the final pseudo-labeled datasets before saving them.












In [ ]:
pubmed_entities_final_df = build_entity_dataframe(
    df=pubmed_sentences_df,
    tag_column="decoded_tags_pseudo_final",
    dataset_name="pubmed"
)

bc5cdr_entities_final_dfs = {}

for split, df in bc5cdr_dfs_pseudo_final.items():
    bc5cdr_entities_final_dfs[split] = build_entity_dataframe(
        df=df,
        tag_column="decoded_tags_pseudo_final",
        dataset_name="bc5cdr",
        split=split
    )

bc5cdr_entities_final_df = pd.concat(
    bc5cdr_entities_final_dfs.values(),
    ignore_index=True
)

print("PubMed final entity DataFrame shape:", pubmed_entities_final_df.shape)
print("BC5CDR final entity DataFrame shape:", bc5cdr_entities_final_df.shape)


## Audit Final Entity Coverage

This section summarizes final entity coverage after both `Gene` and `Virus` pseudo-labeling.

The audit reports how many entity mentions appear for each entity type. For PubMed, the expected final entity types are mainly `Gene` and `Virus`. For BC5CDR, the final entity types should include gold `Chemical` and `Disease` labels as well as pseudo-labeled `Gene` and `Virus` mentions.

This audit provides a high-level check of whether the final label schema has been applied correctly.

In [ ]:

pubmed_final_entity_distribution = (
    pubmed_entities_final_df
    .groupby("entity_type")
    .size()
    .reset_index(name="entity_count")
    .sort_values(by="entity_count", ascending=False)
)

bc5cdr_final_entity_distribution = (
    bc5cdr_entities_final_df
    .groupby(["split", "entity_type"])
    .size()
    .reset_index(name="entity_count")
    .sort_values(by=["split", "entity_count"], ascending=[True, False])
)

print("PubMed final entity distribution:")
display(pubmed_final_entity_distribution)

print("BC5CDR final entity distribution:")
display(bc5cdr_final_entity_distribution)


## Inspect Top Virus Mentions

This section inspects the most frequent `Virus` mentions after final pseudo-labeling.

Top Virus inspection helps verify whether the NCBI Taxonomy dictionary and manual aliases produce biologically meaningful labels. It also helps identify noisy terms that may need to be added to the virus blacklist.


In [ ]:
top_pubmed_virus_mentions = (
    pubmed_entities_final_df[pubmed_entities_final_df["entity_type"] == "Virus"]
    .groupby("entity_text")
    .size()
    .reset_index(name="mention_count")
    .sort_values(by="mention_count", ascending=False)
)

top_pubmed_virus_mentions.head(30)

## Inspect Top Virus Mentions in BC5CDR

This section inspects the most frequent `Virus` pseudo-labels in BC5CDR.

Because BC5CDR is originally focused on `Chemical` and `Disease`, the number of Virus mentions is expected to be relatively small. This inspection checks whether the added Virus labels are plausible and not overly noisy.


In [ ]:

top_bc5cdr_virus_mentions = (
    bc5cdr_entities_final_df[bc5cdr_entities_final_df["entity_type"] == "Virus"]
    .groupby(["split", "entity_text"])
    .size()
    .reset_index(name="mention_count")
    .sort_values(by=["split", "mention_count"], ascending=[True, False])
)

top_bc5cdr_virus_mentions.groupby("split").head(20)

## Audit COVID-19 Virus Blacklist

This section checks whether `COVID-19` is still labeled as a `Virus` after final pseudo-labeling.

This audit is important because `COVID-19` is a disease, not a virus. The correct virus entity is `SARS-CoV-2`.

The expected result is that `COVID-19` should not appear as a `Virus` entity in the final pseudo-labeled datasets.

In [ ]:
def count_entity_text_mentions(entity_df, entity_type, target_text):
    # This function counts exact entity text mentions for a selected entity type
    filtered_df = entity_df[
        (entity_df["entity_type"] == entity_type)
        & (entity_df["entity_text"].str.lower() == target_text.lower())
    ]
    
    return len(filtered_df)


covid19_virus_audit_rows = []

covid19_virus_audit_rows.append({
    "dataset": "pubmed",
    "covid19_as_virus_mentions": count_entity_text_mentions(
        entity_df=pubmed_entities_final_df,
        entity_type="Virus",
        target_text="COVID-19"
    )
})

for split in ["train", "validation", "test"]:
    split_entity_df = bc5cdr_entities_final_df[
        bc5cdr_entities_final_df["split"] == split
    ].copy()
    
    covid19_virus_audit_rows.append({
        "dataset": f"bc5cdr_{split}",
        "covid19_as_virus_mentions": count_entity_text_mentions(
            entity_df=split_entity_df,
            entity_type="Virus",
            target_text="COVID-19"
        )
    })

covid19_virus_audit_df = pd.DataFrame(covid19_virus_audit_rows)

covid19_virus_audit_df

## Audit Gold Label Overwrite After Final Pseudo-Labeling

This section checks whether the final `Gene` and `Virus` pseudo-labeling process overwrote any original BC5CDR gold labels.

BC5CDR gold labels are stored in the original `decoded_tags` column. The final pseudo-labeled sequence is stored in `decoded_tags_pseudo_final`.

The expected result is zero overwritten gold-label tokens. This confirms that the distant supervision process only added pseudo-labels to tokens originally labeled as `O`.

In [ ]:
final_gold_overwrite_rows = []

for split, df in bc5cdr_dfs_pseudo_final.items():
    df_audit = df.copy()
    
    df_audit["gold_token_overwrites"] = df_audit.apply(
        lambda row: count_changed_gold_labels(
            original_tags=row["decoded_tags"],
            pseudo_tags=row["decoded_tags_pseudo_final"]
        ),
        axis=1
    )
    
    final_gold_overwrite_rows.append({
        "split": split,
        "sentences_with_gold_overwrite": (df_audit["gold_token_overwrites"] > 0).sum(),
        "total_gold_token_overwrites": df_audit["gold_token_overwrites"].sum()
    })

final_gold_overwrite_df = pd.DataFrame(final_gold_overwrite_rows)

final_gold_overwrite_df

## Final Audit Summary

At this stage, the final pseudo-labeled datasets have been audited.

The audit checks include:

1. Final entity distribution.
2. Top `Gene` mentions.
3. Top `Virus` mentions.
4. `COVID-19` blacklist validation.
5. BC5CDR gold-label overwrite safety.
6. Token-label alignment validation.

The most important safety check is the BC5CDR gold-label overwrite audit. The expected result is zero overwritten gold-label tokens, because original `Chemical` and `Disease` annotations must remain unchanged.

After these checks, the final pseudo-labeled datasets are ready to be saved for downstream biomedical NER training.

## Save Final Pseudo-Labeled PubMed Dataset

This section saves the final pseudo-labeled PubMed dataset.

The final label column is:

```text
decoded_tags_pseudo_final
```

This column contains the final pseudo-label sequence after both `Gene` and `Virus` distant supervision.

The dataset is saved in two formats:

1. JSONL
   Used as the main format because it preserves list-like columns such as `tokens`, `decoded_tags`, `decoded_tags_pseudo_gene`, and `decoded_tags_pseudo_final`.

2. CSV
   Used for quick inspection and easier manual review.













In [ ]:
pubmed_sentences_df.to_json(
    PUBMED_FINAL_JSONL_PATH,
    orient="records",
    lines=True,
    force_ascii=False
)

pubmed_sentences_df.to_csv(
    PUBMED_FINAL_CSV_PATH,
    index=False
)

print("Saved final PubMed JSONL:", PUBMED_FINAL_JSONL_PATH)
print("Saved final PubMed CSV:", PUBMED_FINAL_CSV_PATH)


## Save Final Pseudo-Labeled BC5CDR Splits

This section saves the final pseudo-labeled BC5CDR train, validation, and test splits.

The original BC5CDR split structure is preserved. Each split contains:

1. Original BC5CDR tokens.
2. Original gold `Chemical` and `Disease` labels in `decoded_tags`.
3. Intermediate `Gene` pseudo-labels in `decoded_tags_pseudo_gene`.
4. Final `Gene` and `Virus` pseudo-labels in `decoded_tags_pseudo_final`.

These files will be used for downstream biomedical NER training.


In [ ]:
for split, df in bc5cdr_dfs_pseudo_final.items():
    output_path = BC5CDR_FINAL_OUTPUT_PATHS[split]
    
    df.to_json(
        output_path,
        orient="records",
        lines=True,
        force_ascii=False
    )
    
    print(f"Saved final BC5CDR {split} split:", output_path)


## Final Dataset Summary

This section creates a compact summary of the final pseudo-labeled datasets.

The summary includes the number of rows for PubMed and each BC5CDR split, as well as the final output paths.


In [ ]:
final_dataset_summary = [
    {
        "dataset": "pubmed",
        "rows": len(pubmed_sentences_df),
        "output_jsonl": PUBMED_FINAL_JSONL_PATH
    }
]

for split, df in bc5cdr_dfs_pseudo_final.items():
    final_dataset_summary.append({
        "dataset": f"bc5cdr_{split}",
        "rows": len(df),
        "output_jsonl": BC5CDR_FINAL_OUTPUT_PATHS[split]
    })

final_dataset_summary_df = pd.DataFrame(final_dataset_summary)

final_dataset_summary_df

## Limitations

The distant supervision approach used in this notebook has several limitations.

First, dictionary-based matching does not understand context. A matched term may be labeled as an entity even when it is used with a different meaning in the sentence.

Second, gene symbols and aliases can be ambiguous. Some terms may overlap with clinical abbreviations, measurement names, disease names, or general biomedical expressions. A manual blacklist is used to reduce obvious false positives, but it cannot remove all noise.

Third, virus names from NCBI Taxonomy can be highly specific. Many names refer to strains, isolates, or reassortants. The filtering step removes overly specific terms, but some noisy or overly broad virus mentions may still remain.

Fourth, the generated `Gene` and `Virus` labels are pseudo-labels, not gold annotations. They should be treated as silver labels for model training and downstream analysis.

Finally, co-occurrence or dictionary-based entity extraction does not directly imply biological causality. Further validation is required before using extracted relations for drug repurposing conclusions.


# Final Notebook Summary

This notebook completed the distant supervision stage for the biomedical NER final project.

The main steps completed in this notebook were:

1. Loaded preprocessed PubMed and BC5CDR datasets.
2. Validated token-label alignment.
3. Built a `Gene` dictionary from HGNC gene symbols, aliases, and previous symbols.
4. Filtered ambiguous Gene terms using stopwords and a manual blacklist.
5. Built a Gene Trie index.
6. Applied `Gene` pseudo-labeling to PubMed and BC5CDR.
7. Audited Gene pseudo-label coverage and gold-label overwrite safety.
8. Built a `Virus` dictionary from NCBI Taxonomy.
9. Filtered noisy and overly specific virus terms.
10. Added manual Virus aliases.
11. Built a Virus Trie index.
12. Applied `Virus` pseudo-labeling on top of the Gene checkpoint.
13. Audited final entity coverage, top Virus mentions, COVID-19 blacklist safety, and BC5CDR gold-label overwrite safety.
14. Saved the final pseudo-labeled datasets.

The final output column is:

```text
decoded_tags_pseudo_final
```

This column contains the final BIO label sequence for the expanded entity schema:

```text
Chemical
Disease
Gene
Virus
```

The resulting datasets are ready for downstream biomedical NER training. After the NER model is trained, extracted entities can be used to construct a biomedical knowledge graph and support drug repurposing analysis.

